# Tema 4 · Matriz de confusión, probabilidades y costos

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

La **Cooperativa de Ahorro y Crédito Ñuble Ahorro** otorga créditos de consumo a sus socios en Chillán. Un proveedor le ofreció un modelo que «acierta el 97 %» al predecir quién dejará de pagar. El comité de crédito desconfía: le parece demasiado bueno, y además no sabe cómo usar el modelo para decidir. Pidió a su equipo revisar qué tan bueno es de verdad un modelo de morosidad y **con qué criterio aprobar o revisar** cada crédito.

> **❓ La pregunta del cliente:** ¿Cómo se sabe si un modelo de morosidad sirve, y con qué criterio conviene usarlo para aprobar créditos?

## 🗂️ Los datos: `t4_creditos.csv` (4.000 créditos de consumo con sus datos y si el socio cayó en mora)

| Columna | Qué significa |
|---|---|
| `credito` | número del crédito |
| `monto_miles` | monto del crédito, en miles de pesos |
| `plazo_meses` | plazo, en meses |
| `ingreso_miles` | ingreso mensual del socio, en miles de pesos |
| `carga_pct` | porcentaje del ingreso que se va en la cuota |
| `atrasos_previos` | atrasos que tuvo en créditos anteriores |
| `antiguedad_anios` | años como socio de la cooperativa |
| `moroso` | **Sí / No**: si dejó de pagar ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t4_creditos.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 4000 filas y 8 columnas.

## 1. Conocer los datos

📘 Cada fila es un **crédito de consumo** que la cooperativa otorgó: el monto y el ingreso del socio (en miles de
pesos), el plazo, la **carga** (qué porcentaje del ingreso se va en la cuota), los atrasos que tenía en créditos
anteriores y sus años como socio. La columna `moroso` dice si después **dejó de pagar**. La celda cuenta cuántos
morosos hay.

In [ ]:
datos.head()

In [ ]:
conteo = datos["moroso"].value_counts()
print("Morosos:", conteo["Sí"], "· Al día:", conteo["No"])
print(f"Proporción de morosos: {conteo['Sí'] / len(datos):.1%}")
# ✅ Debe mostrar: Morosos: 106 · Al día: 3894  /  Proporción de morosos: 2.6%

## 2. Separar datos de entrenamiento y de prueba

📘 El modelo aprende con el 75 % de los créditos y se evalúa con el 25 % restante, que **no vio**. Como los
morosos son tan pocos, `stratify=y` asegura que haya la misma proporción de morosos en las dos partes.

In [ ]:
X = datos[["monto_miles", "plazo_meses", "ingreso_miles", "carga_pct", "atrasos_previos", "antiguedad_anios"]]
y = datos["moroso"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, random_state=2026, stratify=y)
print("Entrenamiento:", len(X_ent), "créditos · Prueba:", len(X_pru), "créditos")
print("Morosos en la prueba:", (y_pru == "Sí").sum())
# ✅ Debe mostrar: Entrenamiento: 3000 créditos · Prueba: 1000 créditos  /  Morosos en la prueba: 27

## 3. ZeroR: el modelo que nunca se equivoca… casi

📘 ZeroR dice siempre la clase más común: **«nadie será moroso»**, es decir, aprobar todos los créditos. La celda
calcula su acierto y cuántos morosos detecta.

In [ ]:
acierto_zeror = (y_pru == "No").mean()
print(f"Acierto de ZeroR: {acierto_zeror:.0%}")
print("Morosos que detecta ZeroR: 0 de", (y_pru == "Sí").sum())
# ✅ Debe mostrar: Acierto de ZeroR: 97%  /  Morosos que detecta ZeroR: 0 de 27

### ✍️ ZeroR acierta casi siempre. ¿Es un buen modelo para la cooperativa? ¿Por qué el porcentaje de acierto engaña en este caso?

_Su respuesta:_

## 4. Un modelo de verdad: regresión logística

📘 La **regresión logística** no responde solo «sí» o «no»: calcula la **probabilidad** de que cada crédito
termine en mora. Después, se decide con un **umbral**: si la probabilidad es mayor o igual que el umbral, el
crédito se marca como riesgoso. El umbral que usan por defecto los programas es **0,5**.
`StandardScaler()` pone las columnas en la misma escala para que el modelo aprenda bien.

In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_ent, y_ent)
prob = modelo.predict_proba(X_pru)[:, list(modelo.classes_).index("Sí")]   # probabilidad de mora
riesgoso = prob >= 0.5
acierto = (riesgoso == (y_pru == "Sí")).mean()
print(f"Acierto con umbral 0,5: {acierto:.0%}")
print("Morosos detectados:", (riesgoso & (y_pru == "Sí")).sum(), "de", (y_pru == "Sí").sum())
# ✅ Debe mostrar: Acierto con umbral 0,5: 98%  /  Morosos detectados: 8 de 27

## 5. La matriz de confusión

📘 La **matriz de confusión** cuenta los cuatro casos posibles. «Positivo» es la clase que interesa detectar: aquí,
**moroso**.

| | El modelo dice moroso | El modelo dice al día |
|---|:-:|:-:|
| **En realidad fue moroso** | **VP** · verdadero positivo: lo detectó | **FN** · falso negativo: se le pasó |
| **En realidad pagó** | **FP** · falso positivo: falsa alarma | **VN** · verdadero negativo |

Con ella se calculan dos medidas que el acierto esconde:
- **Sensibilidad** = VP / (VP + FN): de los morosos reales, ¿qué parte detecta?
- **Precisión** = VP / (VP + FP): de los que marca como morosos, ¿qué parte lo era?

In [ ]:
def matriz(umbral):
    """Cuenta VP, FN, FP y VN de la prueba para un umbral dado."""
    marca = prob >= umbral
    moroso = (y_pru == "Sí").values
    return {"VP": (marca & moroso).sum(), "FN": (~marca & moroso).sum(),
            "FP": (marca & ~moroso).sum(), "VN": (~marca & ~moroso).sum()}

m = matriz(0.5)
print(f"VP = {m['VP']} · FN = {m['FN']}")
print(f"FP = {m['FP']} · VN = {m['VN']}")
print(f"Sensibilidad: {m['VP'] / (m['VP'] + m['FN']):.0%} · Precisión: {m['VP'] / (m['VP'] + m['FP']):.0%}")
# ✅ Debe mostrar: VP = 8 · FN = 19  /  FP = 1 · VN = 972  /  Sensibilidad: 30% · Precisión: 89%

### ✍️ Lean su matriz de confusión con el umbral 0,5. ¿Cuántos morosos se le pasan al modelo? ¿Qué significan su sensibilidad y su precisión para la cooperativa?

_Su respuesta:_

## 6. Mirar las probabilidades

📘 El histograma muestra la probabilidad de mora que el modelo le asignó a cada crédito de prueba, separada según
lo que pasó en realidad. La línea roja es el umbral 0,5.

In [ ]:
plt.figure(figsize=(7, 3.8))
plt.hist(prob[y_pru == "No"], bins=40, range=(0, 1), color="lightgray", label="Pagó")
plt.hist(prob[y_pru == "Sí"], bins=40, range=(0, 1), color="crimson", label="Moroso")
plt.axvline(0.5, color="red", ls="--", label="Umbral 0,5")
plt.yscale("log")
plt.xlabel("Probabilidad de mora calculada por el modelo"); plt.ylabel("Créditos (escala log)")
plt.legend(); plt.title("¿Dónde quedan los morosos?")
plt.show()

## 7. Contar el costo de cada error

📘 Los dos errores **no cuestan lo mismo**:
- **Falso negativo** (aprobar a quien no pagará): la cooperativa pierde el crédito, en promedio **$2.000.000**.
- **Falso positivo** (rechazar a quien sí pagaría): pierde la ganancia de ese crédito, unos **$150.000**.

El **costo total** de un umbral es FN × $2.000.000 + FP × $150.000. Con ZeroR (aprobar a todos) el costo es el
de **todos** los morosos.

In [ ]:
COSTO_FN = 2_000_000     # aprobar a un moroso
COSTO_FP = 150_000       # rechazar a un buen cliente

def costo(umbral):
    m = matriz(umbral)
    return m["FN"] * COSTO_FN + m["FP"] * COSTO_FP

def en_pesos(x):
    """Escribe un monto como $1.234.567."""
    return f"${x:,.0f}".replace(",", ".")

costo_zeror = (y_pru == "Sí").sum() * COSTO_FN
print("Costo de ZeroR (aprobar a todos):", en_pesos(costo_zeror))
print("Costo con umbral 0,5:", en_pesos(costo(0.5)))
# ✅ Debe mostrar: Costo de ZeroR (aprobar a todos): $54.000.000  /  Costo con umbral 0,5: $38.150.000

## 8. Mover el umbral

📘 Bajar el umbral hace al modelo **más desconfiado**: marca como riesgoso a quien tenga una probabilidad de mora
más baja. Detecta más morosos, pero también rechaza a más buenos clientes. ¿Cuál conviene?

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
UMBRAL = 0.5        # ✏️ pruebe con 0.5, 0.3, 0.2, 0.1 y 0.05 (al terminar, deje 0.5)

m = matriz(UMBRAL)
print("Umbral:", UMBRAL)
print("Morosos detectados:", m["VP"], "de", m["VP"] + m["FN"])
print("Buenos clientes rechazados:", m["FP"])
print(f"Acierto: {(m['VP'] + m['VN']) / len(y_pru):.0%}")
print("Costo total:", en_pesos(costo(UMBRAL)))

**✏️ Anoten aquí sus resultados:**

| Umbral | Morosos detectados (de 27) | Buenos clientes rechazados | Acierto | Costo total |
|:-:|:-:|:-:|:-:|:-:|
| 0,5 | | | | |
| 0,3 | | | | |
| 0,2 | | | | |
| 0,1 | | | | |
| 0,05 | | | | |

### ✍️ Según su tabla, ¿qué pasa con el **acierto** y qué pasa con el **costo** a medida que bajan el umbral? ¿Con qué umbral el costo es menor? ¿Por qué no coincide con el umbral de mayor acierto?

_Su respuesta:_

In [ ]:
candidatos = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
costos = [costo(u) for u in candidatos]
plt.figure(figsize=(6.5, 3.8))
plt.plot(candidatos, [c / 1e6 for c in costos], marker="o")
plt.xlabel("Umbral"); plt.ylabel("Costo total (millones de $)")
plt.title("Costo de los errores según el umbral")
plt.show()
print("Umbral de menor costo:", candidatos[costos.index(min(costos))])
# ✅ Debe mostrar: Umbral de menor costo: 0.05

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para el comité de crédito de Ñuble Ahorro: ¿qué umbral usar, cuánto se ahorraría frente a aprobar a todos y qué harían con los socios que el modelo marca como riesgosos?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema4.ipynb`, junto con el informe en PDF.